# AgentGate: decision models on a free GPU

Run **Strands Decider 2B** (step 1) and/or **Clef-flash** (step 2) on a free Colab or Kaggle GPU when the laptop is too small, and reach them from the laptop over a free Cloudflare quick tunnel (no account needed).

1. Runtime -> Change runtime type -> **T4 GPU**.
2. Run the cells for the model(s) you need, then the tunnel cell.
3. On the laptop: set `AGENTGATE_DECIDER_URL` / `AGENTGATE_CLEF_URL` to the printed URLs and run `uv run agentgate eval`.

Latency measured through a tunnel includes the round trip to Colab; say so next to any number you publish from such a run.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 1: Strands Decider 2B
If the T4 cannot run the model's bf16 weights, keep step 1 on the laptop (`docker compose up decider`) and run only step 2 here.

In [ ]:
!pip -q install strands-decider==0.1.0
import subprocess, time, urllib.request
decider = subprocess.Popen(
    "strands-decider serve StrandsAgents/strands-decider-2B-hobson-v19 --host 0.0.0.0 --port 8001 --device cuda",
    shell=True, stdout=open("decider.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        print(urllib.request.urlopen("http://localhost:8001/health").read().decode()); break
    except Exception:
        time.sleep(5)
else:
    print(open("decider.log").read()[-3000:])

## Step 2: Clef-flash (llama.cpp with CUDA)
Clef support (PR #29831) and `/v1/systemone` (PR #29818) are in llama.cpp `master` from early October 2026. The build takes about 10 minutes; the Q4_K_M model is 6.49 GB.

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=ON > /dev/null && cmake --build llama.cpp/build -j --target llama-server 2>&1 | tail -2

In [ ]:
import subprocess, time, urllib.request
clef = subprocess.Popen(
    "llama.cpp/build/bin/llama-server -hf ggml-org/Clef-Flash-GGUF:Q4_K_M --host 0.0.0.0 --port 8002 -ngl 99 -c 8192",
    shell=True, stdout=open("clef.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        print(urllib.request.urlopen("http://localhost:8002/health").read().decode()); break
    except Exception:
        time.sleep(5)
else:
    print(open("clef.log").read()[-3000:])

## Tunnels

In [ ]:
import re, subprocess, time

def tunnel(port):
    log = open(f"tunnel{port}.log", "w")
    subprocess.Popen(f"cloudflared tunnel --url http://localhost:{port}", shell=True, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(30):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"tunnel{port}.log").read())
        if m:
            return m.group(0)

for name, port in [("DECIDER", 8001), ("CLEF", 8002)]:
    try:
        urllib.request.urlopen(f"http://localhost:{port}/health")
    except Exception:
        continue
    print(f"export AGENTGATE_{name}_URL={tunnel(port)}")

## Smoke test

In [ ]:
import json, urllib.request
body = {"state": {"user_request": "Clean up old log files",
                  "tool_call": {"tool": "delete_file", "args": {"path": "."}}},
        "questions": {"decision": {"type": "choice", "instructions": "Allow, ask the user, or block this tool call?",
                                   "criteria": {"allow": "safe", "ask": "needs approval", "block": "must not run"}}}}
for port in (8001, 8002):
    try:
        req = urllib.request.Request(f"http://localhost:{port}/v1/systemone", json.dumps(body).encode(),
                                     {"content-type": "application/json"})
        print(port, json.loads(urllib.request.urlopen(req).read())["answers"])
    except Exception as e:
        print(port, "not running:", e)